# Final Findings Summary
## BNPL merchant ranking project

This notebook summarises the main analytical findings from the BNPL merchant-onboarding project.
The objective is to produce a reproducible ranking of the merchant population, identify the top
100 overall, examine the top merchants within behavioural segments, and incorporate fraud risk
without overstating what the sparse fraud labels can support.


All commercial and customer features use March 2021-September 2022. The September evaluation rebuilds pre-September features, but September is inspected by multiple experiments; it is not an independent model-selection holdout. Counts and recommendations below are calculated from current outputs.


## Stage 1: Loading and cleaning

The cleaning pipeline standardises merchant, consumer, transaction and fraud tables, removes
invalid core transaction records, and handles duplicate `order_id` values deterministically.
Potential outliers are investigated rather than automatically removed because the transaction
distribution is naturally right-skewed and high-value purchases can be valid for particular
merchant types.


In [1]:
WEIGHTS_V1 = {
    "estimated_bnpl_revenue": .30, "log_value_growth": .20,
    "transactions_per_customer": .10, "stability": .10,
    "unique_customers": .05, "active_month_ratio": .05, "repeat_rate": .05,
    "customer_income": .05, "customer_irsad": .05, "customer_spread": .05,
}
NAMES = {1: "Established", 3: "Rising", 2: "Fading", 0: "Intermittent"}


# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession, Window, functions as F

# Locate the repository by its data/tables directory rather than by folder name.
def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "tables").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing data/tables/."
    )

PROJECT_ROOT = find_project_root()
PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
if not PROCESSED_PATH.is_dir():
    raise FileNotFoundError(f"{PROCESSED_PATH} does not exist.")

RANDOM_STATE = 42
spark = SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").config("spark.driver.memory", "6g").getOrCreate()

import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, PowerTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import boxcox

from scipy.stats import norm
AMOUNT_CUTOFF = 8000.0
MAX_PLOT_ROWS = 100_000
RANDOM_STATE = 42

# Shared look for the distribution plots: light blue histogram, thin red KDE line.
HIST_BLUE, KDE_RED = "#9ecae1", "#d62728"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 23:30:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
transactions = spark.read.parquet(
    str(PROCESSED_PATH / "transactions_clean")
)

merchants = spark.read.parquet(
    str(PROCESSED_PATH / "merchants_clean")
)

consumers = spark.read.parquet(
    str(PROCESSED_PATH / "consumers")
)

duplicates = spark.read.parquet(
    str(PROCESSED_PATH / "duplicate_order_ids")
)

n_tx = transactions.count()
tx_merchants = transactions.select("merchant_abn").distinct()
tx_users = transactions.select("user_id").distinct()

print("Duplicate order rows:", n_tx - transactions.select("order_id").distinct().count())
print("Transacting merchants with no metadata:",
      tx_merchants.join(merchants, "merchant_abn", "left_anti").count())
print("Registered merchants that never transact:",
      merchants.join(tx_merchants, "merchant_abn", "left_anti").count())
print("Transacting users missing from consumers:",
      tx_users.join(consumers, "user_id", "left_anti").count())
print("Registered consumers that never transact:",
      consumers.join(tx_users, "user_id", "left_anti").count())


Duplicate order rows: 0


Transacting merchants with no metadata: 396


Registered merchants that never transact: 0


Transacting users missing from consumers: 0


Registered consumers that never transact: 475918


## Outlier analysis and distributions

For an idea on the shape of numerical data we now have a look at some of the distributional plots of our (collected) dataset.

In [3]:
value = F.col("dollar_value").cast("double")
valid = value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")) & (value >= 0)
cost = transactions.select(value.alias("dollar_value")).filter(valid)
valid_n = cost.count()

# Bound driver-memory use; plots use a seeded Spark sample, capped at MAX_PLOT_ROWS.
fraction = min(1.0, MAX_PLOT_ROWS / valid_n)
plot_values = (cost.sample(False, fraction, seed=RANDOM_STATE).limit(MAX_PLOT_ROWS)
               .toPandas()["dollar_value"].to_numpy())

log_cost = cost.withColumn("log_cost", F.log1p("dollar_value"))
log_values = np.log1p(plot_values)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(x=log_values, bins=120, stat="density", color=HIST_BLUE, edgecolor="none",
             alpha=0.7, ax=ax)
sns.kdeplot(x=log_values, color=KDE_RED, linewidth=1, ax=ax)
ticks = np.arange(0, int(np.ceil(log_values.max())) + 1)
ax.set_xticks(ticks, [f"{np.expm1(t):,.0f}" for t in ticks], rotation=45)
ax.set(xlabel="Transaction amount ($; log1p spacing)", ylabel="Density",
       title="Log-transformed amount distribution")
sns.despine(ax=ax)
fig.tight_layout()
plt.show()


/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/1700129218.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Market segments

Merchants were grouped with k-means (k = 4) on six features from `merchant_features`, each
converted to a percentile rank and standardised before clustering (`scripts/segment_behaviour.py`).

| Dimension | Column | In the hybrid ranking? |
|---|---|---|
| Commercial value | `estimated_bnpl_revenue` | yes, 70% |
| Customer reach | `customer_penetration` | no — Hybrid uses customer_spread instead, at 5% |
| Engagement | `transactions_per_customer` | no, 0% |
| Growth | `log_value_growth` | yes, 2% |
| Historical activity | `active_month_ratio` | yes, 18% |
| Volatility | `value_cv` | no, 0% |

`customer_penetration` is calculated against the 24,081 customers active in the complete-month
production window (March 2021–September 2022).


In [4]:
# Estimated BNPL revenue per merchant, by behaviour segment (scripts/segment_behaviour.py).
COL = "estimated_bnpl_revenue"
NAMES = {1: "Established", 3: "Rising", 2: "Fading", 0: "Intermittent"}

segments = pd.read_parquet(PROCESSED_PATH / "merchant_behaviour_segments")[["merchant_abn", "segment_id"]]
ranking = pd.read_parquet(PROCESSED_PATH / "merchant_ranking_hybrid.parquet")
seg_df = ranking.merge(segments, on="merchant_abn", how="inner")

# Revenue spans several orders of magnitude, so bin and smooth on log10 dollars.
log_rev = np.log10(seg_df[COL])
bins = np.linspace(log_rev.min(), log_rev.max(), 81)

fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True)
for ax, seg_id in zip(axes, [1, 3, 2, 0]):
    block = log_rev[seg_df["segment_id"] == seg_id]
    sns.histplot(x=block, bins=bins, stat="density", color=HIST_BLUE, edgecolor="none",
                 alpha=0.7, ax=ax)
    sns.kdeplot(x=block, color=KDE_RED, linewidth=1, ax=ax)
    ax.set_title(f"{NAMES[seg_id]}  (n = {len(block):,}, median ${10 ** block.median():,.0f})",
                 loc="left", fontsize=10)
    ax.set_ylabel("Density")
    sns.despine(ax=ax)
ticks = np.arange(np.floor(log_rev.min()), np.ceil(log_rev.max()) + 1)
axes[-1].set_xticks(ticks, [f"${10 ** t:,.0f}" for t in ticks])
axes[-1].set_xlabel("Estimated BNPL revenue, Mar 2021-Sep 2022 (log scale)")
fig.tight_layout()
plt.show()


/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/181695943.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Top 10 merchants within each behavioural segment

The same frozen hybrid feature weights are used in every segment, but percentile ranks are recalculated
within each segment. This produces a peer-relative shortlist without fitting segment-specific
weights from future outcomes.

The table below is generated by `scripts/top10_per_segment.py`.


In [5]:
segment_top10_path = PROCESSED_PATH / "top10_per_segment.csv"
if not segment_top10_path.exists():
    raise FileNotFoundError(
        f"{segment_top10_path} is missing - run scripts/top10_per_segment.py first."
    )

segment_top10 = pd.read_csv(segment_top10_path)

segment_top10[
    [
        "segment_name",
        "rank_in_segment",
        "name",
        "segment_score",
        "estimated_bnpl_revenue",
        "rank_overall",
    ]
].sort_values(
    ["segment_name", "rank_in_segment"]
).round(2)


,segment_name,rank_in_segment,name,segment_score,estimated_bnpl_revenue,rank_overall
10,Established,1,Dignissim Maecenas Foundation,87.62,590673.67,1
11,Established,2,Odio Phasellus Institute,87.59,536062.14,9
12,Established,3,Arcu Sed Eu Incorporated,87.55,432454.15,3
13,Established,4,Leo In Consulting,87.48,581136.25,2
14,Established,5,Mauris Non Institute,87.41,544064.20,4
15,Established,6,Lacus Consulting,87.35,555429.64,5
16,Established,7,Orci In Consequat Corporation,87.29,590013.60,7
17,Established,8,Ornare Limited,87.29,548590.52,10
18,Established,9,Lobortis Ultrices Company,87.20,529710.79,6
19,Established,10,Est Nunc Consulting,87.12,506253.31,8


## Ranking model

Each rankable merchant receives a score from 0 to 100 in `merchant_ranking_hybrid.ipynb`:

1. **Every feature becomes a percentile rank.** This allows quantities with different units to be
   combined without letting a few extreme dollar values dominate the scale.
2. **Thin evidence is pulled towards the middle.** Revenue and growth ranks are shrunk towards 50
   using `n / (n + 300)`, where `n` is the merchant's transaction count.
3. **Missing features receive a neutral score of 50.**
4. **The score is a weighted sum**, with historical BNPL revenue carrying 70% of the final hybrid
   weight.

The first version spread weight more evenly across ten features. March–August 2022 was then used
as a **development window** to diagnose v1 and choose the revised hybrid design. September 2022 is
evaluated with history-only predictors; September is also inspected by supporting experiments, so the development window is not presented as final
out-of-sample evidence.


In [6]:
# v1 weights as set in merchant_ranking_baseline.ipynb; hybrid weights as saved by merchant_ranking_hybrid.ipynb.
WEIGHTS_V1 = {"estimated_bnpl_revenue": 0.30, "log_value_growth": 0.20, "transactions_per_customer": 0.10,
              "stability": 0.10, "unique_customers": 0.05, "active_month_ratio": 0.05, "repeat_rate": 0.05,
              "customer_income": 0.05, "customer_irsad": 0.05, "customer_spread": 0.05}
weights_hybrid = pd.read_csv(PROCESSED_PATH / "weights_hybrid.csv", index_col=0)["weight"]

weights = pd.DataFrame({"v1": pd.Series(WEIGHTS_V1), "hybrid (shipped)": weights_hybrid}).fillna(0) * 100
weights = weights.sort_values("hybrid (shipped)", ascending=False)
weights_long = (weights.rename_axis("feature").reset_index()
                .melt("feature", var_name="version", value_name="weight"))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=weights_long, y="feature", x="weight", hue="version",
            palette=["#b0b0b0", "#4C78A8"], ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.0f%%", padding=3, fontsize=8)
ax.set(xlabel="Weight in the score (%)", ylabel="",
       title="Ranking weights before and after the time-split validation")
ax.legend(frameon=False, loc="lower right", title=None)
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/1831836224.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Does it predict?

March-August 2022 is the development period. The September comparison rebuilds
all predictors, including customer profiles, from history ending 31 August.
October is excluded from both production and September feature construction.

September is also inspected by learned and rolling comparisons. This is a
history-only evaluation, not an independently untouched model-selection test.
The table is loaded from hybrid_holdout_results.csv; compare every method with
the past-revenue benchmark and retain the evaluation limitations.


In [7]:
holdout_path = PROCESSED_PATH / "hybrid_holdout_results.csv"
if not holdout_path.exists():
    raise FileNotFoundError(
        f"{holdout_path} is missing - rerun merchant_ranking_hybrid.ipynb through the "
        "September holdout section first."
    )

holdout_results = pd.read_csv(holdout_path, index_col=0)
holdout_results.round(3)


,spearman,capture rate,hits in realised top 100,picks with no September activity,realised revenue ($M)
picker,,,,,
v1 (original score),0.740,0.645,46,0,1.244
hybrid (frozen before holdout),0.907,0.911,80,0,1.757
baseline: past revenue,0.940,1.000,99,0,1.928
perfect foresight,1.000,1.000,100,0,1.929


**Interpretation.** Compare hybrid with the past-revenue baseline on the untouched September
holdout. If revenue-only remains stronger, the multi-factor score should be defended for the
additional growth, stability, customer-concentration and risk context it provides — not by
claiming higher predictive accuracy. The model is left frozen after this result is observed.


In [8]:
# The top of the shipped list (before the fraud penalty).
RANK_COLS = ["rank_overall", "name", "score", "take_rate", "estimated_bnpl_revenue",
             "transaction_count", "unique_customers", "log_value_growth"]
ranking.nsmallest(10, "rank_overall")[RANK_COLS].round(2)

,rank_overall,name,score,take_rate,estimated_bnpl_revenue,transaction_count,unique_customers,log_value_growth
0,1,Dignissim Maecenas Foundation,89.41,6.64,590673.67,62814,22326,0.36
1,2,Leo In Consulting,89.39,6.43,581136.25,258319,24081,0.34
2,3,Arcu Sed Eu Incorporated,89.38,4.80,432454.15,35930,18686,0.36
3,4,Mauris Non Institute,89.35,6.10,544064.20,114164,23872,0.35
4,5,Lacus Consulting,89.33,6.98,555429.64,216173,24080,0.34
5,6,Lobortis Ultrices Company,89.24,6.31,529710.79,107421,23816,0.34
6,7,Orci In Consequat Corporation,89.24,6.61,590013.60,81192,23266,0.33
7,8,Est Nunc Consulting,89.24,6.01,506253.31,204232,24077,0.34
8,9,Odio Phasellus Institute,89.19,6.59,536062.14,10826,8678,0.39
9,10,Ornare Limited,89.19,5.91,548590.52,29496,17009,0.36


The complete production shortlist is written by
`merchant_ranking_hybrid.ipynb` to `data/tables/processed/top100_hybrid.csv`. The table above shows only
the first ten rows for readability.


## Fraud: models to understand the scores, arithmetic to price them

The supplied fraud scores cover under 1% of transactions (34,765 consumer user-days and 114 merchant-days), and none is below 8. They are a pre-screened sample, so we did not impute the missing ones. The fraud work has two separate parts:

**1. Models were used to understand why a score is given.** XGBoost classifiers were trained to predict whether a scored day is flagged (score ≥ 30). They are diagnostic: they show what the scores respond to. They are **not** an input to the ranking.

- Consumer scores follow the **size of the transaction**.
- Merchant scores follow how large the day is **relative to that merchant's usual day**.

The cell below loads the trained models saved by `fraud_models.ipynb` (in `data/tables/models/`) and plots their predictions on held-out data against that single driver.

**2. The penalty uses the supplied scores directly, with no model.** For every transaction in the scoring window:

- $p_\text{consumer}$ = consumer score ÷ 100, split across that user's purchases that day in proportion to dollar value
- $p_\text{merchant}$ = merchant score ÷ 100
- $P(\text{fraud}) = 1 - (1 - p_\text{merchant})(1 - p_\text{consumer})$
- expected loss $= P(\text{fraud}) \times$ transaction value

Scores below 30 count as zero. Summing per merchant gives a loss rate, which is subtracted from the merchant's BNPL revenue before it is ranked. So the penalty depends on how well calibrated the supplied scores are, not on the accuracy of any model we trained.


In [9]:
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

# Trained models and their evaluation data are exported by fraud_models.ipynb (sections 10 and 13).
MODEL_DIR = PROJECT_ROOT / "data" / "tables" / "models"
FLAG_CUTOFF = 30   # a score at or above this is treated as a fraud flag


def load_model(name):
    path = MODEL_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing - run the notebook that trains and saves it first.")
    model = XGBClassifier()
    model.load_model(path)
    return model


def top_features(model, n=4):
    gain = pd.Series(model.get_booster().get_score(importance_type="gain"))
    return ", ".join(f"{name} ({share:.0%})" for name, share in (gain / gain.sum()).nlargest(n).items())


# --- Consumer side: the saved XGBoost model applied to users it never saw in training.
consumer_model = load_model(f"consumer_fraud_ge{FLAG_CUTOFF}")
consumer_test = pd.read_parquet(MODEL_DIR / "consumer_fraud_test.parquet")
consumer_test["flagged"] = (consumer_test["fraud_probability"] >= FLAG_CUTOFF).astype(int)
consumer_test["p_flagged"] = consumer_model.predict_proba(
    consumer_test[consumer_model.get_booster().feature_names])[:, 1]
# Days with one transaction, where the score belongs to exactly that transaction.
consumer_single = consumer_test[consumer_test["day_txn_count"] == 1]

# --- Merchant side: 114 scored days. The saved model was fitted on all of them, so the
# predictions plotted are the out-of-fold ones (each merchant predicted by a model
# trained on the other 60), exported alongside it.
merchant_model = load_model(f"merchant_fraud_ge{FLAG_CUTOFF}")
merchant_days = pd.read_parquet(MODEL_DIR / "merchant_fraud_days.parquet")
merchant_days["flagged"] = (merchant_days["fraud_probability"] >= FLAG_CUTOFF).astype(int)
merchant_days["p_flagged"] = merchant_days[f"p_ge_{FLAG_CUTOFF}_oof"]

for label, frame, model in [("consumer", consumer_single, consumer_model),
                            ("merchant", merchant_days, merchant_model)]:
    print(f"{label}: {len(frame):,} held-out scored days, {frame['flagged'].sum()} flagged | "
          f"PR-AUC {average_precision_score(frame['flagged'], frame['p_flagged']):.2f} "
          f"(random = {frame['flagged'].mean():.2f})")
    print(f"  model leans on: {top_features(model)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
panels = [
    (axes[0], consumer_single, "dollar_value", 10, "Transaction value ($, log scale)",
     "Consumer: flagged when the purchase is large"),
    (axes[1], merchant_days, "revenue_ratio", 30, "Day's revenue ÷ merchant's median day (log scale)",
     "Merchant: flagged when the day is a spike"),
]
for ax, frame, x, size, xlabel, title in panels:
    for flagged, colour, name in [(0, HIST_BLUE, f"score < {FLAG_CUTOFF}"), (1, KDE_RED, f"score ≥ {FLAG_CUTOFF}")]:
        part = frame[frame["flagged"] == flagged]
        ax.scatter(part[x], part["p_flagged"], s=size, color=colour, alpha=0.6, edgecolor="none", label=name)
    ax.set_xscale("log")
    ax.set(xlabel=xlabel, title=title)
    sns.despine(ax=ax)
axes[0].set_ylabel(f"XGBoost predicted P(score ≥ {FLAG_CUTOFF}), held out")
axes[0].legend(frameon=False, loc="upper left", title="Actual score")
fig.tight_layout()
plt.show()

consumer: 1,853 held-out scored days, 135 flagged | PR-AUC 0.97 (random = 0.07)
  model leans on: value_vs_user_median (36%), log_value (30%), merchant_median_value (9%), month (5%)
merchant: 114 held-out scored days, 73 flagged | PR-AUC 0.98 (random = 0.64)
  model leans on: revenue_ratio (43%), avg_value_ratio (16%), revenue_z (10%), max_share_of_day (6%)


/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/3340197926.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Fraud penalty: what it changes in the ranking

Compares the hybrid ranking (no fraud) with the fraud-adjusted ranking from `fraud_penalty.ipynb`. `score_change` is the fraud-adjusted score minus the hybrid score, so the most negative values are the merchants the penalty hit hardest.


In [10]:
fraud = pd.read_parquet(PROCESSED_PATH / "merchant_ranking_hybrid_fraud_adjusted.parquet")
fraud["score_change"] = fraud["score_adjusted"] - fraud["score"]
penalised = fraud["expected_fraud_loss"] > 0

print(f"{penalised.sum():,} of {len(fraud):,} merchants have some expected fraud loss; "
      f"{(fraud['score_change'] < -1e-9).sum():,} end up with a lower score")

# The merchants the penalty moved furthest down.
FRAUD_COLS = ["name", "rank_overall", "rank_adjusted", "score", "score_adjusted", "score_change",
              "estimated_bnpl_revenue", "expected_fraud_loss", "loss_to_revenue",
              "avg_transaction_value", "transaction_count", "risk_tier"]
fraud.nsmallest(15, "score_change")[FRAUD_COLS].round(2)

1,117 of 4,026 merchants have some expected fraud loss; 245 end up with a lower score


,name,rank_overall,rank_adjusted,score,score_adjusted,score_change,estimated_bnpl_revenue,expected_fraud_loss,loss_to_revenue,avg_transaction_value,transaction_count,risk_tier
4025,Lacus Aliquam Corporation,307,4026,80.83,23.27,-57.57,385503.71,1613157.97,4.18,6308.81,1448,watch
4024,Commodo Ipsum Industries,530,4025,76.03,25.78,-50.26,382918.02,1725653.39,4.51,9919.67,783,watch
4018,Pharetra Quisque Company,706,4019,72.43,28.99,-43.44,138710.75,1422423.40,10.25,10027.09,528,watch
3441,Odio Incorporated,1065,3442,65.36,39.31,-26.04,138710.07,915832.35,6.60,16141.84,186,flagged
3556,Nec Quam Corp.,1184,3557,62.90,38.41,-24.49,43647.51,280885.19,6.44,6813.11,208,none
3426,Fusce Incorporated,1165,3427,63.23,39.45,-23.78,50628.97,231790.53,4.58,7588.62,189,none
3578,Ligula Elit Pretium Foundation,1232,3579,61.97,38.25,-23.72,149486.21,1318421.09,8.82,19383.59,160,flagged
3603,Faucibus Leo Corp.,1265,3604,61.30,38.02,-23.29,49074.63,225438.47,4.59,7389.09,185,watch
3248,Malesuada Augue Ut Associates,1166,3249,63.17,40.65,-22.51,57445.05,182354.56,3.17,6784.42,168,none
3860,Eget Tincidunt Dui LLC,1429,3861,58.36,36.18,-22.18,39700.92,43522.05,1.10,4487.81,194,none


In [11]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=fraud[~penalised], x="avg_transaction_value", y="score_change", s=8,
                color="#b0b0b0", edgecolor="none", alpha=0.5, label="no expected loss", ax=ax)
sns.scatterplot(data=fraud[penalised], x="avg_transaction_value", y="score_change", s=14,
                color=KDE_RED, edgecolor="none", alpha=0.7, label="has expected fraud loss", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xscale("log")
ax.set(xlabel="Merchant's average transaction value ($, log scale)",
       ylabel="Score change (fraud-adjusted - hybrid)",
       title="Fraud penalty by merchant ticket size")
ax.legend(frameon=False, loc="lower left")
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/4188544551.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Fraud scores: the scoring changes part-way through, and three levels of caution

The supplied scores do not behave the same way across their own window. The plots below are the raw scores, with nothing imputed; the working is in `fraud_imputation_check.ipynb` §9.

In [12]:
import matplotlib.dates as mdates

SCORE_START, SCORE_END = "2021-02-28", "2022-02-27"   # fraud scores exist only in this window
REGIME_START = "2021-08-28"                           # first day of the second transaction snapshot
CUTOFFS = [30, 50, 70]                                # lowest score counted as a fraud signal: three levels of caution
BINS_PER_DECADE = 10                                  # log-spaced bands of a user's total spend that day
MAX_SPEND_BIN = int(np.floor(np.log10(20_000) * BINS_PER_DECADE))   # pool everything above about $20K
OBS_BLUE, EARLY_ORANGE, IMP_RED = "#4C78A8", "#F58518", KDE_RED
BREAK = pd.Timestamp(REGIME_START)


def p_if_at_least(score_col, min_score):
    return F.when(F.col(score_col) >= min_score, F.col(score_col) / 100).otherwise(0.0)


consumer_scores = (spark.read.parquet(str(PROCESSED_PATH / "consumer_fraud_probability"))
                   .where(F.col("fraud_probability").isNotNull())
                   .select("user_id", F.to_date("order_datetime").alias("order_date"),
                           F.col("fraud_probability").alias("consumer_score")))
# One row per user per day: that day's total spend, and the supplied score where there is one.
user_days = (
    transactions.withColumn("order_date", F.to_date("order_datetime"))
    .groupBy("user_id", "order_date").agg(F.sum("dollar_value").alias("day_spend"))
    .join(consumer_scores, ["user_id", "order_date"], "left")
    .withColumn("spend_bin", F.least(F.floor(F.log10("day_spend") * BINS_PER_DECADE), F.lit(MAX_SPEND_BIN)))
).cache()

scored_days = user_days.where(F.col("consumer_score").isNotNull()).toPandas()
scored_days["order_date"] = pd.to_datetime(scored_days["order_date"])
merchant_raw = (spark.read.parquet(str(PROCESSED_PATH / "merchant_fraud_probability"))
                .select(F.to_date("order_datetime").alias("order_date"),
                        F.col("fraud_probability").alias("merchant_score")).toPandas())
merchant_raw["order_date"] = pd.to_datetime(merchant_raw["order_date"])

per_day = scored_days.groupby("order_date").size().reindex(pd.date_range(SCORE_START, SCORE_END), fill_value=0)
fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True, height_ratios=[3, 1.6, 2])
axes[0].scatter(scored_days["order_date"], scored_days["consumer_score"], s=4, color=OBS_BLUE,
                alpha=0.25, linewidths=0)
axes[0].set(ylabel="Fraud probability (%)", ylim=(0, 100),
            title=f"Consumer scores: all {len(scored_days):,} supplied")
axes[1].plot(per_day.index, per_day.values, color=OBS_BLUE, linewidth=1.2)
axes[1].set(ylabel="Scores per day", title="Consumer scores supplied per day")
axes[1].set_ylim(0, None)
axes[2].scatter(merchant_raw["order_date"], merchant_raw["merchant_score"], s=22, color=OBS_BLUE,
                alpha=0.7, linewidths=0)
axes[2].set(ylabel="Fraud probability (%)", ylim=(0, 100),
            title=f"Merchant scores: all {len(merchant_raw)} supplied")
for ax in axes:
    ax.axvline(BREAK, color="#555555", linewidth=1, linestyle="--")
    ax.set_xlim(pd.Timestamp(SCORE_START), pd.Timestamp(SCORE_END))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.grid(color="#ececec")
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
axes[0].text(BREAK, 96, " 28 Aug 2021: second snapshot begins", fontsize=8, color="#555555", va="top")
fig.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/4269410560.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


On 28 Aug 2021, the first day of the second transaction snapshot, consumer scores go from about 25 a day to about 150, and merchant scores from 2 in six months to 112 in the next six. Spending does not change across that date (about \$3.7M a day either side).

The next plot shows what changed. Each dot is a scored user-day, placed by how much the user spent that day.

In [13]:
late_scored = scored_days[scored_days["order_date"] >= BREAK]
early_scored = scored_days[scored_days["order_date"] < BREAK]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, divisor, title in [(axes[0], 1, "As recorded"), (axes[1], 2, "Early-window days drawn at half their spend")]:
    ax.scatter(late_scored["day_spend"], late_scored["consumer_score"], s=4, color=OBS_BLUE, alpha=0.15,
               linewidths=0, label="late window (28 Aug 2021 onwards)")
    ax.scatter(early_scored["day_spend"] / divisor, early_scored["consumer_score"], s=4, color=EARLY_ORANGE,
               alpha=0.35, linewidths=0, label="early window (before 28 Aug 2021)")
    ax.set_xscale("log")
    ax.set_xticks([1500, 3000, 6000, 12000, 25000, 50000], ["$1.5k", "$3k", "$6k", "$12k", "$25k", "$50k"])
    ax.minorticks_off()
    ax.set(xlabel="User's total spend that day", title=title, xlim=(1200, 80_000), ylim=(0, 100))
    ax.grid(color="#ececec")
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
axes[0].set_ylabel("Fraud probability (%)")
legend = axes[0].legend(frameon=False, fontsize=8, markerscale=3)
for handle in legend.legend_handles:
    handle.set_alpha(1)
fig.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/2979860339.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


- **Before 28 Aug 2021, a day is scored as if the user had spent half as much.** Drawn at half their spend, the early scores fall on the later relationship.
- **Days with a single purchase are never scored before that date,** whatever their size. Afterwards 99% of days of \$3,000 or more are.

A change this exact, on a file boundary, with spending unchanged, is a change in how the scores were produced, not in how much fraud there was. It does not tell us which of the two rules is right. We take the later one as the true level: it has complete coverage, it is the rule in force when the scores end, and it gives the more cautious loss figure.

### Expected fraud loss through time, at three cutoffs

A score only counts as a fraud signal if it is at or above a cutoff. Three are shown, from most to least cautious: **30, 50 and 70**.

- **Blue** is the loss the supplied scores give.
- **Red** is the imputed loss: each user-day is given the average fraud probability of its spend band, measured on 28 Aug 2021 – 27 Feb 2022 where every large day carries a score. It is drawn across the whole dataset so it can be compared with the real scores where both exist.

Loss is the fraud probability × the day's spend (full principal lost), as a 7-day rolling figure per \$100 transacted.

In [14]:
late_days = user_days.where(F.col("order_date").between(REGIME_START, SCORE_END))
lookup = late_days.groupBy("spend_bin").agg(
    *[(F.sum(p_if_at_least("consumer_score", c) * F.col("day_spend")) / F.sum("day_spend")).alias(f"p_day_min{c}")
      for c in CUTOFFS])

daily = (
    user_days.join(F.broadcast(lookup), "spend_bin", "left")
    .groupBy("order_date")
    .agg(F.sum("day_spend").alias("value"),
         *[F.sum(p_if_at_least("consumer_score", c) * F.col("day_spend")).alias(f"observed_loss_min{c}")
           for c in CUTOFFS],
         *[F.sum(F.coalesce(F.col(f"p_day_min{c}"), F.lit(0.0)) * F.col("day_spend")).alias(f"imputed_loss_min{c}")
           for c in CUTOFFS])
    .toPandas()
)
daily["order_date"] = pd.to_datetime(daily["order_date"])
daily = daily.sort_values("order_date").set_index("order_date")
in_window = (daily.index >= SCORE_START) & (daily.index <= SCORE_END)
rolling = daily.rolling("7D").sum()

SPANS = [(SCORE_START, REGIME_START, OBS_BLUE, 0.07, "real scores, early rule"),
         (REGIME_START, "2022-02-28", OBS_BLUE, 0.17, "real scores, later rule"),
         ("2022-02-28", "2022-10-27", IMP_RED, 0.08, "no scores: imputed only")]
fig, axes = plt.subplots(len(CUTOFFS), 1, figsize=(11, 9.5), sharex=True, sharey=True)
for ax, c in zip(axes, CUTOFFS):
    for start, end, colour, alpha, name in SPANS:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=colour, alpha=alpha, linewidth=0, label=name)
    imputed = rolling[f"imputed_loss_min{c}"] / rolling["value"] * 100
    observed = (rolling[f"observed_loss_min{c}"] / rolling["value"] * 100).where(in_window)
    ax.plot(imputed.index, imputed, color=IMP_RED, linewidth=1.3, label="imputed")
    ax.plot(observed.index, observed, color=OBS_BLUE, linewidth=1.8, label="real scores")
    ax.set(ylabel="Expected loss per \$100 ($)", title=f"Counting only scores of {c} and above")
    ax.set_xlim(pd.Timestamp(SCORE_START), pd.Timestamp("2022-10-27"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.grid(color="#ececec")
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
axes[0].set_ylim(0, None)
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, fontsize=9, ncol=5,
           loc="upper center", bbox_to_anchor=(0.5, 1.02))
fig.tight_layout()
plt.show()

# Totals over the whole dataset (every transaction, 28 Feb 2021 - 26 Oct 2022).
late = (daily.index >= REGIME_START) & (daily.index <= SCORE_END)
totals = pd.DataFrame({
    "real scores only ($M)": [daily[f"observed_loss_min{c}"].sum() / 1e6 for c in CUTOFFS],
    "real in the late window, imputed elsewhere ($M)":
        [(daily.loc[late, f"observed_loss_min{c}"].sum() + daily.loc[~late, f"imputed_loss_min{c}"].sum()) / 1e6
         for c in CUTOFFS],
}, index=pd.Index(CUTOFFS, name="cutoff"))
totals["share of value transacted"] = totals.iloc[:, 1] * 1e6 / daily["value"].sum()
print("Expected consumer-fraud loss, all transactions:")
print(totals.round(3).to_string())

Expected consumer-fraud loss, all transactions:
        real scores only ($M)  real in the late window, imputed elsewhere ($M)  share of value transacted
cutoff                                                                                                   
30                     19.017                                           49.546                      0.021
50                     10.052                                           27.848                      0.012
70                      3.287                                            9.015                      0.004


/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27503/3560874730.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Overall takeaway

The final analysis keeps three questions separate:

- **commercial ranking:** the hybrid score produces the final multi-factor Top 100 shortlist;
- **peer comparison:** the same frozen hybrid priorities are recalibrated within each behavioural
  segment to produce the Top 10 per segment;
- **benchmarking:** historical BNPL revenue remains the simple predictive reference and performs
  better than hybrid on the September evaluation;
- **risk review:** sparse fraud scores and merchant-risk flags modify or review the shortlist but
  are not treated as perfectly calibrated fraud probabilities.

Taken together, the notebook now reports the overall ranking, the segment shortlists, the untouched
September validation and the fraud-adjusted sensitivity analysis with their assumptions and
limitations stated explicitly.
